# Pruebas clasificacion de olas

In [ ]:
from pyspark.sql import functions as F
env = 'project'

## Seleccionar los datos

In [ ]:
data = (
    spark.sql(
        f"""
            SELECT coast_name, datetime, wind_u, wind_v, wave_u, wave_v, wave_period_s
            FROM cor_{env}.silver.swell_metrics
        """
    )
)
data_pd = data.toPandas()

In [ ]:
data_pre_processing= (
    data
    .withColumn('coast_year_month', F.concat(F.col('coast_name'), F.lit('_'), F.date_format('datetime', 'yyyy-MM')))
)

coast_year_month_dict = {row.coast_year_month: 0.1 for row in data_pre_processing.select('coast_year_month').distinct().collect()}
data_sample = (
    data_pre_processing
    .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=0)
    .drop('coast_year_month')
).toPandas()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
fig = make_subplots(rows=2, cols=1, subplot_titles=(
    "wave_u vs wave_v total vs sample",
    "wind_u vs wind_v total vs sample"
))
fig.add_trace(
    go.Scatter(x=data_pd['wave_u'], y=data_pd['wave_v'], mode='markers', name='Total', marker=dict(color='blue', size=5, opacity=0.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=data_sample['wave_u'], y=data_sample['wave_v'], mode='markers', name='Sample', marker=dict(color='red', size=5, opacity=0.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=data_pd['wind_u'], y=data_pd['wind_v'], mode='markers', name='Total', marker=dict(color='blue', size=5, opacity=0.5)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=data_sample['wind_u'], y=data_sample['wind_v'], mode='markers', name='Sample', marker=dict(color='red', size=5, opacity=0.5)),
    row=2, col=1
)
fig.update_layout(height=800, width=600, title_text="Comparación de Distribuciones: Total vs Muestra")
fig.to_html("comparacion_distribuciones.html")

## Preparar los datos

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from pyspark.sql import functions as F

In [ ]:
features = ['wind_u', 'wind_v', 'wave_u', 'wave_v', 'wave_period_s']

scaler = StandardScaler()
X = scaler.fit_transform(data_sample[features])
X['coast_name'] = data_sample['coast_name']
X['datetime'] = data_sample['datetime']

In [ ]:
from sklearn.metrics import silhouette_score as sil_score
import numpy as np
import itertools

In [ ]:
posible_e = np.linspace(0.01, 1, 15)